---
**Maju Bareng AI Project**

**Nama  :** Adhit Hikmatullah

---

# Install Library

In [1]:
# Install library
!pip install -q streamlit pandas pyngrok python-dotenv
print("Library berhasil diinstall!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.8 MB/s eta 0:00:00
Library berhasil diinstall!


# Set Ngrok Auth Token

In [2]:
import os

from pyngrok import ngrok
from google.colab import userdata

# Token dari Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
os.environ["NGROK_TOKEN"] = NGROK_TOKEN
print("Token berhasil dimuat dari Colab Secrets!")

Token berhasil dimuat dari Colab Secrets!


# Upload File Katalog Buku

In [3]:
from google.colab import files
import shutil

print("Silakan upload file  kamu:")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = "Files"
    with open(dest, "wb") as f:
        f.write(data)
    print(f"File '{fname}' berhasil diupload")


Silakan upload file  kamu:


Saving catalog_buku.csv to catalog_buku.csv
File 'catalog_buku.csv' berhasil diupload


# Buat File

In [4]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
import os

# Page Config
st.set_page_config(
    page_title="Katalog Buku",
    page_icon="📚",
    layout="wide",
    initial_sidebar_state="collapsed",
)

# Load CSV
CSV_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "catalog_buku.csv")

@st.cache_data
def load_catalog(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={
        "nama_buku":     "judul",
        "detail_produk": "format",
        "harga_lama":    "harga_lama",
        "harga_diskon":  "harga_diskon",
        "tanggal_terbit":"tanggal_terbit",
    })
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    for col in ["judul", "penulis", "format", "tanggal_terbit"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    df["diskon_persen"] = (
        (df["harga_lama"] - df["harga_diskon"]) / df["harga_lama"] * 100
    ).round(1)
    df["hemat"] = df["harga_lama"] - df["harga_diskon"]

    bulan_order = {
        "Februari 2025": 1, "Maret 2025": 2, "April 2025": 3,
        "Mei 2025": 4,      "Juni 2025": 5,  "Juli 2025": 6,
        "Agustus 2025": 7,
    }
    df["bulan_order"] = df["tanggal_terbit"].map(bulan_order).fillna(0).astype(int)
    return df.reset_index(drop=True)

df = load_catalog(CSV_PATH)

# Helpers
def fmt_rp(val: int) -> str:
    return "Rp {:,}".format(int(val)).replace(",", ".")

# System Prompt
@st.cache_data
def build_catalog_str(df: pd.DataFrame) -> str:
    lines = []
    for _, r in df.iterrows():
        disc = round((r["harga_lama"] - r["harga_diskon"]) / r["harga_lama"] * 100, 1)
        lines.append(
            f"- {r['judul']} | Penulis: {r['penulis']} | Format: {r['format']} | "
            f"Terbit: {r['tanggal_terbit']} | "
            f"Harga normal: Rp{int(r['harga_lama']):,} | "
            f"Harga diskon: Rp{int(r['harga_diskon']):,} | Diskon: {disc}%"
        )
    return "\n".join(lines)

# Session State
if "messages" not in st.session_state: st.session_state.messages = []
if "page" not in st.session_state: st.session_state.page = 0
if "prev_filt" not in st.session_state: st.session_state.prev_filt = ""

CARDS_PER_PAGE = 10   # 5 rows × 2 cols

# Header

st.markdown("""
<div class='app-header'>
    <h1>📚 Katalog Buku</h1>
</div>
""", unsafe_allow_html=True)

# Statistic Cards

cheapest  = df.loc[df["harga_diskon"].idxmin()]
most_disc = df.loc[df["diskon_persen"].idxmax()]
latest    = df.sort_values("bulan_order", ascending=False).iloc[0]["tanggal_terbit"]
n_latest  = len(df[df["tanggal_terbit"].str.strip() == latest.strip()])

st.markdown(f"""
<div class='stat-row'>
    <div class='stat-card'>
        <div class='s-label'>📚 Total Buku</div>
        <div class='s-val'>{len(df)} Judul</div>
        <div class='s-sub'>{df["penulis"].nunique()} penulis berbeda</div>
    </div>
    <div class='stat-card'>
        <div class='s-label'>💰 Rata-rata Diskon</div>
        <div class='s-val'>{df["diskon_persen"].mean():.1f}%</div>
        <div class='s-sub'>Hemat rata-rata {fmt_rp(int(df["hemat"].mean()))}</div>
    </div>
    <div class='stat-card'>
        <div class='s-label'>🏷️ Buku Termurah</div>
        <div class='s-val'>{fmt_rp(int(cheapest["harga_diskon"]))}</div>
        <div class='s-sub'>{cheapest["judul"][:32]}{"…" if len(cheapest["judul"])>32 else ""}</div>
    </div>
    <div class='stat-card'>
        <div class='s-label'>🔥 Diskon Terbesar</div>
        <div class='s-val'>{most_disc["diskon_persen"]}%</div>
        <div class='s-sub'>{most_disc["judul"][:32]}{"…" if len(most_disc["judul"])>32 else ""}</div>
    </div>
    <div class='stat-card'>
        <div class='s-label'>🆕 Update Terbaru</div>
        <div class='s-val'>{latest}</div>
        <div class='s-sub'>{n_latest} buku baru ditambahkan</div>
    </div>
</div>
""", unsafe_allow_html=True)

# Main Layout
col_catalog, col_chat = st.columns([1.1, 0.9], gap="large")

## Katalog Buku
with col_catalog:
    st.markdown("<div class='sec-header'>🗂️ Katalog Buku</div>", unsafe_allow_html=True)

    ### Filter Bar
    with st.container():
        st.markdown("<div class='filter-bar'>", unsafe_allow_html=True)
        fa, fb, fc = st.columns([1, 1, 1.2])
        with fa:
            fmt_opts = ["Semua Format"] + sorted(df["format"].dropna().unique().tolist())
            sel_fmt  = st.selectbox("📦 Format", fmt_opts, key="sel_fmt")
        with fb:
            bln_opts = (["Semua Bulan"] +
                df[["tanggal_terbit","bulan_order"]].drop_duplicates()
                  .sort_values("bulan_order")["tanggal_terbit"].tolist()
            )
            sel_bln = st.selectbox("📅 Bulan Terbit", bln_opts, key="sel_bln")
        with fc:
            search_q = st.text_input("🔍 Cari judul / penulis",
                                     placeholder="ketik judul atau nama penulis...",
                                     key="search_q")
        st.markdown("</div>", unsafe_allow_html=True)

    ### Slider Harga
    p_min = int(df["harga_diskon"].min())
    p_max = int(df["harga_diskon"].max())
    price_range = st.slider(
        "💰 Range Harga Diskon",
        min_value=p_min, max_value=p_max,
        value=(p_min, p_max),
        step=5000, format="Rp %d", key="price_range",
    )

    ### Apply filters
    fil = df.copy()
    if sel_fmt != "Semua Format":
        fil = fil[fil["format"] == sel_fmt]
    if sel_bln != "Semua Bulan":
        fil = fil[fil["tanggal_terbit"].str.strip() == sel_bln.strip()]
    if search_q.strip():
        q = search_q.strip().lower()
        fil = fil[
            fil["judul"].str.lower().str.contains(q, na=False) |
            fil["penulis"].str.lower().str.contains(q, na=False)
        ]
    fil = fil[(fil["harga_diskon"] >= price_range[0]) & (fil["harga_diskon"] <= price_range[1])]

    ### Reset pagination
    filt_key = f"{sel_fmt}|{sel_bln}|{search_q}|{price_range}"
    if filt_key != st.session_state.prev_filt:
        st.session_state.page      = 0
        st.session_state.prev_filt = filt_key

    total_books = len(fil)
    total_pages = max(1, -(-total_books // CARDS_PER_PAGE))
    cur_page    = min(st.session_state.page, total_pages - 1)

    st.markdown(
        f"<div class='result-count'>"
        f"Menampilkan <b style='color:#e8c97a'>{total_books}</b> buku  •  "
        f"Halaman {cur_page + 1}/{total_pages}"
        f"</div>",
        unsafe_allow_html=True,
    )

    # Book Grid
    page_df = fil.iloc[cur_page * CARDS_PER_PAGE : (cur_page + 1) * CARDS_PER_PAGE]

    if total_books == 0:
        st.info("Tidak ada  buku yang cocok dengan  filter ini, coba ganti filter-nya")
    else:
        for i in range(0, len(page_df), 2):
            row_slice = page_df.iloc[i : i + 2]
            gcols = st.columns(2)
            for gi, (_, r) in enumerate(row_slice.iterrows()):
                with gcols[gi]:
                    st.markdown(f"""
                    <div class='book-card'>
                        <div class='bc-title'>{r["judul"]}</div>
                        <div class='bc-author'>✍️ {r["penulis"]}</div>
                        <div class='bc-meta'>
                            <span class='bc-format'>{r["format"]}</span>
                            <span class='bc-date'>🗓️ {r["tanggal_terbit"]}</span>
                        </div>
                        <div class='bc-footer'>
                            <div>
                                <div class='bc-price-old'>{fmt_rp(int(r["harga_lama"]))}</div>
                                <div class='bc-price-disc'>{fmt_rp(int(r["harga_diskon"]))}</div>
                            </div>
                            <span class='bc-badge'>-{r["diskon_persen"]}%</span>
                        </div>
                    </div>
                    """, unsafe_allow_html=True)

        # Pagination
        if total_pages > 1:
            pg1, pg2, pg3 = st.columns([1, 2, 1])
            with pg1:
                if cur_page > 0:
                    if st.button("← Prev", use_container_width=True, key="btn_prev"):
                        st.session_state.page = cur_page - 1
                        st.rerun()
            with pg2:
                st.markdown(
                    f"<div class='pager-info'>Halaman {cur_page+1} dari {total_pages}</div>",
                    unsafe_allow_html=True,
                )
            with pg3:
                if cur_page < total_pages - 1:
                    if st.button("Next →", use_container_width=True, key="btn_next"):
                        st.session_state.page = cur_page + 1
                        st.rerun()

Writing /content/app.py


# Streamlit

In [6]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# Terminate any previous ngrok tunnels
ngrok.kill()

# Konfigurasi Ngrok
conf.get_default().auth_token = NGROK_TOKEN
STREAMLIT_PORT = 8501

# Streamlit di background
def run_streamlit():
    subprocess.run(
        [
            "streamlit",
            "run",
            "app.py",
            "--server.port",
            str(STREAMLIT_PORT),
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
            "--browser.gatherUsageStats",
            "false",
            "--logger.level",
            "error",
        ],
        env={**__import__('os').environ},
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Ngrok tunnel
tunnel = ngrok.connect(STREAMLIT_PORT, "http")
public_url = tunnel.public_url

print(f"  Buka di browser  : {public_url}")
print(f"   Lokal Colab      : http://localhost:{STREAMLIT_PORT}")

  Buka di browser  : https://pox-procreate-flashy.ngrok-free.dev
   Lokal Colab      : http://localhost:8501
